In [2]:
from urllib.request import urlopen
from pathlib import Path
import requests
import time
import pandas as pd
import json


/opt/miniconda3/envs/strava/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
working_directory = Path.cwd().parent.parent
print(working_directory)

file_path = working_directory / "data" / "clean" / "race_session_meeting_info.csv"
race_sessions = pd.read_csv(file_path)

/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


### getting every meeting key
- need to use these to get the session keys for each of the races

In [4]:
meeting_keys = []

for key in race_sessions["meeting_key"].unique():
    meeting_keys.append(int(key))


print(meeting_keys)

[1141, 1142, 1143, 1207, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1226, 1229, 1230, 1231, 1232, 1233, 1234, 1235, 1236, 1237, 1238, 1239, 1240, 1241, 1242, 1243, 1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1252, 1254, 1255, 1256, 1257, 1258, 1259, 1260, 1261, 1262, 1263, 1264, 1277, 1265, 1266, 1267, 1268, 1269, 1270, 1271, 1272, 1273, 1274, 1275, 1276, 1279, 1280, 1281, 1282, 1283, 1284, 1285, 1286, 1287, 1288, 1289, 1290, 1291, 1292, 1293, 1294, 1295, 1296, 1297, 1298, 1299, 1300, 1301, 1302]


### function that will give us a dataframe of each of the sessions of each of the meeting keys

In [5]:
def get_session_info_from_meeting(meeting_key, max_retries = 10, sleep_seconds=2):
    url = "https://api.openf1.org/v1/sessions"
    params = {
        "meeting_key": meeting_key
    }

    #throttling call
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)


        #not a valid session_key
        if response.status_code == 404:
            raise requests.exceptions.HTTPError(
                f"404 not found for session_key:{session_key}"
            )
        
        #rate limit hit
        if response.status_code == 429:
            wait = sleep_seconds * (attempt +1)
            print(f"Rate Limited on session_key={session_key}. Sleeping {wait} seconds")
            time.sleep(wait)
            continue
            
        response.raise_for_status()
        return pd.DataFrame(response.json())
    
    raise requests.exceptions.HTTPError(
        f"429 persisted after {max_retries} retries for session_key={session_key}",
        response=response
    )

In [6]:
def get_session_info(year, max_retries = 10, sleep_seconds=2):
    url = "https://api.openf1.org/v1/sessions"
    params = {
        "year": year
    }

    #throttling call
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)


        #not a valid session_key
        if response.status_code == 404:
            raise requests.exceptions.HTTPError(
                f"404 not found for session_key:{session_key}"
            )
        
        #rate limit hit
        if response.status_code == 429:
            wait = sleep_seconds * (attempt +1)
            print(f"Rate Limited on session_key={session_key}. Sleeping {wait} seconds")
            time.sleep(wait)
            continue
            
        response.raise_for_status()
        return pd.DataFrame(response.json())
    
    raise requests.exceptions.HTTPError(
        f"429 persisted after {max_retries} retries for session_key={session_key}",
        response=response
    )

In [7]:
sessions_2026 = get_session_info(2026)

In [8]:
#deprecated
# def get_session_info(meeting_key):
#     url = "https://api.openf1.org/v1/sessions"
#     params = {
#         "meeting_key": meeting_key
#     }

#     response = requests.get(url, params=params, timeout=30)
#     response.raise_for_status()

#     df = pd.DataFrame(response.json())
#     df.head()

#     return df

In [9]:
get_session_info(2023)

,session_key,session_type,session_name,date_start,date_end,meeting_key,circuit_key,circuit_short_name,country_key,country_code,country_name,location,gmt_offset,year
0,9222,Practice,Day 1,2023-02-23T07:00:00+00:00,2023-02-23T16:30:00+00:00,1140,63,Sakhir,36,BRN,Bahrain,Sakhir,03:00:00,2023
1,7763,Practice,Day 2,2023-02-24T07:00:00+00:00,2023-02-24T16:30:00+00:00,1140,63,Sakhir,36,BRN,Bahrain,Sakhir,03:00:00,2023
2,7764,Practice,Day 3,2023-02-25T07:00:00+00:00,2023-02-25T16:30:00+00:00,1140,63,Sakhir,36,BRN,Bahrain,Sakhir,03:00:00,2023
3,7765,Practice,Practice 1,2023-03-03T11:30:00+00:00,2023-03-03T12:30:00+00:00,1141,63,Sakhir,36,BRN,Bahrain,Sakhir,03:00:00,2023
4,7766,Practice,Practice 2,2023-03-03T15:00:00+00:00,2023-03-03T16:00:00+00:00,1141,63,Sakhir,36,BRN,Bahrain,Sakhir,03:00:00,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,9190,Practice,Practice 1,2023-11-24T09:30:00+00:00,2023-11-24T10:30:00+00:00,1226,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Island,04:00:00,2023
114,9191,Practice,Practice 2,2023-11-24T13:00:00+00:00,2023-11-24T14:00:00+00:00,1226,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Island,04:00:00,2023
115,9192,Practice,Practice 3,2023-11-25T10:30:00+00:00,2023-11-25T11:30:00+00:00,1226,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Island,04:00:00,2023
116,9193,Qualifying,Qualifying,2023-11-25T14:00:00+00:00,2023-11-25T15:00:00+00:00,1226,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Island,04:00:00,2023


### Function that will give us a dataframe of the placements for a given meeting key

In [10]:
def get_race_session_result(session_key, max_retries=10, sleep_seconds=2):

    url = "https://api.openf1.org/v1/session_result"
    params = {
        "session_key": session_key
    }

    #throttling call
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)


        #not a valid session_key
        if response.status_code == 404:
            raise requests.exceptions.HTTPError(
                f"404 not found for session_key:{session_key}"
            )
        
        #rate limit hit
        if response.status_code == 429:
            wait = sleep_seconds * (attempt +1)
            print(f"Rate Limited on session_key={session_key}. Sleeping {wait} seconds")
            time.sleep(wait)
            continue
            
        response.raise_for_status()
        return pd.DataFrame(response.json())
    
    raise requests.exceptions.HTTPError(
        f"429 persisted after {max_retries} retries for session_key={session_key}",
        response=response
    )

### Function that will give me the driver details for a given driver number and session key
- this will help us build a table that has driver name

In [11]:
def get_driver_info(session_key, max_retries=10, sleep_seconds=2):
    
    #package
    url = "https://api.openf1.org/v1/drivers"
    params = {
        
        "session_key": session_key
    }
    
    #throttligg call
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)


        #not a valid session_key
        if response.status_code == 404:
            raise requests.exceptions.HTTPError(
                f"404 not found for session_key:{session_key}"
            )
        
        #rate limit hit
        if response.status_code == 429:
            wait = sleep_seconds * (attempt +1)
            print(f"Rate Limited on session_key={session_key}. Sleeping {wait} seconds")
            time.sleep(wait)
            continue
            
        response.raise_for_status()
        return pd.DataFrame(response.json())
    
    raise requests.exceptions.HTTPError(
        f"429 persisted after {max_retries} retries for session_key={session_key}",
        response=response
    )


##### example of getting bahrain 2023 driver info

In [12]:
get_driver_info(7953)

,meeting_key,session_key,driver_number,broadcast_name,full_name,name_acronym,team_name,team_colour,first_name,last_name,headshot_url,country_code
0,1141,7953,1,M VERSTAPPEN,Max VERSTAPPEN,VER,Red Bull Racing,3671C6,Max,Verstappen,https://www.formula1.com/content/dam/fom-websi...,NED
1,1141,7953,2,L SARGEANT,Logan SARGEANT,SAR,Williams,37BEDD,Logan,Sargeant,https://www.formula1.com/content/dam/fom-websi...,USA
2,1141,7953,4,L NORRIS,Lando NORRIS,NOR,McLaren,F58020,Lando,Norris,https://www.formula1.com/content/dam/fom-websi...,GBR
3,1141,7953,10,P GASLY,Pierre GASLY,GAS,Alpine,2293D1,Pierre,Gasly,https://www.formula1.com/content/dam/fom-websi...,FRA
4,1141,7953,11,S PEREZ,Sergio PEREZ,PER,Red Bull Racing,3671C6,Sergio,Perez,https://www.formula1.com/content/dam/fom-websi...,MEX
5,1141,7953,14,F ALONSO,Fernando ALONSO,ALO,Aston Martin,358C75,Fernando,Alonso,https://www.formula1.com/content/dam/fom-websi...,ESP
6,1141,7953,16,C LECLERC,Charles LECLERC,LEC,Ferrari,F91536,Charles,Leclerc,https://www.formula1.com/content/dam/fom-websi...,MON
7,1141,7953,18,L STROLL,Lance STROLL,STR,Aston Martin,358C75,Lance,Stroll,https://www.formula1.com/content/dam/fom-websi...,CAN
8,1141,7953,20,K MAGNUSSEN,Kevin MAGNUSSEN,MAG,Haas F1 Team,B6BABD,Kevin,Magnussen,https://www.formula1.com/content/dam/fom-websi...,DEN
9,1141,7953,21,N DE VRIES,Nyck DE VRIES,DEV,AlphaTauri,5E8FAA,Nyck,De Vries,https://www.formula1.com/content/dam/fom-websi...,NED


### Single Race Pull, to test out functions and joins

In [13]:
years = [2023, 2024, 2025, 2026]
all_results = []
exception_session_keys = []

for year in years:

    # need to get the session key for the race

    session_info_df = get_session_info(year)
    session_info_df = session_info_df[session_info_df["session_name"] == "Race"]
    session_info_df[["session_key", "date_start", "meeting_key", "circuit_key", "location"]]

    #getting each race session key for the given year
    race_session_keys = []

    for session_key in session_info_df["session_key"].unique():
        race_session_keys.append(int(session_key))



    for session_key in race_session_keys:
        try:
            #session_key = race_session_keys[0:5] #session key will eventually be an iterable
            curr_race_result = get_race_session_result(session_key)

            #need to add to this data set the driver acronym
            curr_session_driver_info = get_driver_info(session_key)
            curr_session_driver_info = curr_session_driver_info[["driver_number", "name_acronym", "meeting_key", "session_key"]]
            
            #merge the driver info to the curr race result
            curr_race_result = curr_race_result.merge(curr_session_driver_info, how="left", on=["driver_number", "meeting_key", "session_key"]).rename(columns={"name_acronym": "driver"})
            
            #cut down to only needed columns
            curr_race_result = curr_race_result[["date_start", "driver", "driver_number", "position", "points", "gap_to_leader", "meeting_key", "session_key"]]

            all_results.append(curr_race_result)

            print(f"ADDED session_key {session_key}")
        
        except Exception as e:
            print(f"SKIPPING session_key {session_key}: {e}")
            exception_session_keys.append(session_key)
            continue

placements = pd.concat(all_results, ignore_index=True)





SKIPPING session_key 7953: "['date_start'] not in index"
SKIPPING session_key 7779: "['date_start'] not in index"
SKIPPING session_key 7787: "['date_start'] not in index"
SKIPPING session_key 9070: "['date_start'] not in index"
SKIPPING session_key 9078: "['date_start'] not in index"
SKIPPING session_key 9086: 404 not found for session_key:9086
SKIPPING session_key 9094: "['date_start'] not in index"


KeyboardInterrupt: 

#### Verifying that all of the exception keys are 

In [1]:
sessions_2026[sessions_2026["session_key"].isin(exception_session_keys)].shape

NameError: name 'sessions_2026' is not defined

In [194]:
len(exception_session_keys)

24

In [195]:
exception_session_keys

[9086,
 11245,
 11253,
 11261,
 11269,
 11280,
 11291,
 11299,
 11307,
 11315,
 11326,
 11334,
 11342,
 11353,
 11361,
 11369,
 11377,
 11388,
 11396,
 11404,
 11412,
 11420,
 11428,
 11436]

In [196]:
placements.shape

(1419, 7)

In [197]:
placements.head()

,driver,driver_number,position,points,gap_to_leader,meeting_key,session_key
0,VER,1,1.0,25.0,0,1141,7953
1,PER,11,2.0,18.0,11.987,1141,7953
2,ALO,14,3.0,15.0,38.637,1141,7953
3,SAI,55,4.0,12.0,48.052,1141,7953
4,HAM,44,5.0,10.0,50.977,1141,7953
